# ETL and Preprocessing

## Цель
Загрузить и очистить 4 файла датасета, подготовить данные для анализа.

## Аналитические вопросы
1. Какие категории товаров формируют основную выручку?
2. Какие сегменты клиентов наиболее подвержены оттоку?
3. Как меняются метрики (AOV, retention) по годам?

## Ожидаемый результат
- Очищенные таблицы: `customers_clean.csv`, `orders_clean.csv`, `products_clean.csv`, `revenue_monthly_clean.csv`
- Отчёт по качеству данных

In [1]:
#Импорт библиотек

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Настройки отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.rcParams['figure.figsize'] = (12, 6)

In [2]:
# Загрузка данных
# Пути к файлам
DATA_DIR = '../data/raw/'

# Загрузка
customers = pd.read_csv(f'{DATA_DIR}customers.csv')
orders = pd.read_csv(f'{DATA_DIR}orders.csv')
product_summary = pd.read_csv(f'{DATA_DIR}product_summary.csv')
monthly_revenue = pd.read_csv(f'{DATA_DIR}monthly_revenue.csv')

# Единый словарь для всех последующих проверок
frames = {
    'customers': customers,
    'orders': orders,
    'product_summary': product_summary,
    'monthly_revenue': monthly_revenue,
}

print(f"customers: {customers.shape}")
print(f"orders: {orders.shape}")
print(f"product_summary: {product_summary.shape}")
print(f"monthly_revenue: {monthly_revenue.shape}")



customers: (8000, 20)
orders: (25000, 28)
product_summary: (140, 9)
monthly_revenue: (75, 10)


##Проверка структуры данных

In [3]:
# Схема и размерность
for name, df in frames.items():
    print(f"{name}: {df.shape}")
    print(f"  columns: {list(df.columns)}")
    print()
    # Первые строки — визуальная проверка форматов значений
    display(df.head(3))

customers: (8000, 20)
  columns: ['customer_id', 'country', 'age', 'gender', 'membership_tier', 'registration_date', 'total_orders', 'total_spend_usd', 'avg_order_value_usd', 'days_since_last_purchase', 'preferred_category', 'preferred_device', 'preferred_payment_method', 'acquisition_channel', 'reviews_given', 'avg_review_score', 'returns_made', 'wishlist_items', 'newsletter_subscribed', 'churned']



,customer_id,country,age,gender,membership_tier,registration_date,total_orders,total_spend_usd,avg_order_value_usd,days_since_last_purchase,preferred_category,preferred_device,preferred_payment_method,acquisition_channel,reviews_given,avg_review_score,returns_made,wishlist_items,newsletter_subscribed,churned
0,C00001,United States,40,Male,Free,2019-01-17,4,286.63,63.78,49,Food & Grocery,Mobile,Debit Card,Social Media,1,4.5,0,12,0,0
1,C00002,United States,20,Female,Free,2026-03-04,11,1245.18,107.32,126,Toys & Games,Mobile,Debit Card,Organic Search,2,2.6,1,1,0,0
2,C00003,United States,43,Female,Gold,2026-02-08,4,195.37,42.74,0,Home & Kitchen,Mobile,PayPal,Referral,0,4.8,0,0,1,0


orders: (25000, 28)
  columns: ['order_id', 'customer_id', 'order_date', 'year', 'month', 'quarter', 'day_of_week', 'product_name', 'category', 'unit_price_usd', 'quantity', 'subtotal_usd', 'discount_pct', 'discount_amount_usd', 'shipping_fee_usd', 'tax_pct', 'tax_amount_usd', 'total_amount_usd', 'payment_method', 'device_used', 'delivery_days', 'delivery_date', 'order_status', 'returned', 'customer_rating', 'session_duration_minutes', 'pages_viewed_before_purchase', 'is_repeat_customer']



,order_id,customer_id,order_date,year,month,quarter,day_of_week,product_name,category,unit_price_usd,quantity,subtotal_usd,discount_pct,discount_amount_usd,shipping_fee_usd,tax_pct,tax_amount_usd,total_amount_usd,payment_method,device_used,delivery_days,delivery_date,order_status,returned,customer_rating,session_duration_minutes,pages_viewed_before_purchase,is_repeat_customer
0,O000001,C07108,2020-08-27,2020,8,Q3,Thursday,Tire Inflator,Automotive,62.91,1,62.91,0,0.00,0.00,18,11.32,74.23,Credit Card,Desktop,3,2020-08-30,Delivered,0,NaN,14.4,1,1
1,O000002,C03487,2024-04-11,2024,4,Q2,Thursday,Stud Earrings Gold,Jewelry & Accessories,18.44,1,18.44,0,0.00,3.99,10,1.84,24.27,Credit Card,Mobile,2,2024-04-13,Delivered,0,NaN,9.0,4,0
2,O000003,C03062,2023-06-25,2023,6,Q2,Sunday,Pen Set Premium,Office Supplies,109.79,1,109.79,20,21.96,0.00,8,7.03,94.86,PayPal,Desktop,3,2023-06-28,Delivered,0,4.0,3.2,17,0


product_summary: (140, 9)
  columns: ['category', 'product_name', 'total_orders', 'total_revenue_usd', 'avg_price', 'avg_rating', 'return_rate', 'avg_discount_pct', 'avg_delivery_days']



,category,product_name,total_orders,total_revenue_usd,avg_price,avg_rating,return_rate,avg_discount_pct,avg_delivery_days
0,Automotive,Air Freshener,87,10543.38,73.61,3.95,7.0,4.54,4.66
1,Automotive,Car Phone Mount,86,10950.18,73.82,3.99,12.0,5.76,4.08
2,Automotive,Car Vacuum Cleaner,73,8588.32,71.15,4.11,7.0,7.53,4.38


monthly_revenue: (75, 10)
  columns: ['year', 'month', 'quarter', 'orders', 'revenue_usd', 'avg_order_value', 'avg_discount_pct', 'return_rate', 'unique_customers', 'new_customers']



,year,month,quarter,orders,revenue_usd,avg_order_value,avg_discount_pct,return_rate,unique_customers,new_customers
0,2020,1,Q1,266,35415.25,133.14,5.68,0.0,261,100
1,2020,2,Q1,268,34304.78,128.00,6.06,0.0,263,93
2,2020,3,Q1,258,32642.23,126.52,5.21,0.0,254,81


In [4]:
#Типы данных

for name, df in frames.items():
    print(f"{name}:")
    print(df.dtypes)
    print()

customers:
customer_id                     str
country                         str
age                           int64
gender                          str
membership_tier                 str
registration_date               str
total_orders                  int64
total_spend_usd             float64
avg_order_value_usd         float64
days_since_last_purchase      int64
preferred_category              str
preferred_device                str
preferred_payment_method        str
acquisition_channel             str
reviews_given                 int64
avg_review_score            float64
returns_made                  int64
wishlist_items                int64
newsletter_subscribed         int64
churned                       int64
dtype: object

orders:
order_id                            str
customer_id                         str
order_date                          str
year                              int64
month                             int64
quarter                             str
day_of

## Проверка качества данных

In [5]:
# Сводка качества: пропуски и дубликаты
quality = pd.DataFrame({
    'rows':      {k: v.shape[0] for k, v in frames.items()},
    'cols':      {k: v.shape[1] for k, v in frames.items()},
    'missing':   {k: int(v.isna().sum().sum()) for k, v in frames.items()},
    'missing_%': {k: round(v.isna().sum().sum() / v.size * 100, 2) for k, v in frames.items()},
    'dupes':     {k: int(v.duplicated().sum()) for k, v in frames.items()},
})
quality

,rows,cols,missing,missing_%,dupes
customers,8000,20,0,0.00,0
orders,25000,28,15749,2.25,0
product_summary,140,9,0,0.00,0
monthly_revenue,75,10,0,0.00,0


In [6]:
# Числовые статистики

for name, df in frames.items():
    print(f"{name}")
    display(df.describe())
    print()

customers


,age,total_orders,total_spend_usd,avg_order_value_usd,days_since_last_purchase,reviews_given,avg_review_score,returns_made,wishlist_items,newsletter_subscribed,churned
count,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000
mean,35.616375,16.545250,1558.642350,94.845566,59.583875,3.228750,4.109112,0.849500,4.457125,0.617375,0.089375
std,11.170455,14.681064,2284.094953,78.992885,60.610355,3.942698,0.523992,1.407337,4.854391,0.486058,0.285302
min,18.000000,1.000000,4.890000,5.000000,0.000000,0.000000,1.800000,0.000000,0.000000,0.000000,0.000000
25%,27.000000,5.000000,336.055000,44.690000,16.000000,0.000000,3.800000,0.000000,1.000000,0.000000,0.000000
50%,35.000000,12.000000,845.700000,72.270000,41.000000,2.000000,4.200000,0.000000,3.000000,1.000000,0.000000
75%,43.000000,23.000000,1892.165000,118.560000,84.000000,5.000000,4.500000,1.000000,6.000000,1.000000,0.000000
max,75.000000,79.000000,61282.480000,1051.730000,582.000000,28.000000,5.000000,11.000000,41.000000,1.000000,1.000000



orders


,year,month,unit_price_usd,quantity,subtotal_usd,discount_pct,discount_amount_usd,shipping_fee_usd,tax_pct,tax_amount_usd,total_amount_usd,delivery_days,returned,customer_rating,session_duration_minutes,pages_viewed_before_purchase,is_repeat_customer
count,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,9251.000000,25000.000000,25000.00000,25000.000000
mean,2022.652840,6.352280,68.124030,1.695520,116.187322,5.630000,6.345500,3.867579,10.677080,11.746785,125.456186,4.179480,0.080800,4.002324,16.780432,6.51256,0.645960
std,1.814996,3.514614,57.258933,1.045436,136.994998,9.740785,17.530764,3.269618,6.504922,17.723056,145.635016,2.548507,0.272533,0.577576,15.811830,5.26557,0.478231
min,2020.000000,1.000000,3.360000,1.000000,3.360000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000,1.000000,0.000000,1.500000,0.300000,1.00000,0.000000
25%,2021.000000,3.000000,29.250000,1.000000,38.467500,0.000000,0.000000,0.000000,8.000000,2.300000,43.450000,3.000000,0.000000,3.600000,7.100000,2.00000,0.000000
50%,2023.000000,6.000000,51.530000,1.000000,72.390000,0.000000,0.000000,3.990000,10.000000,6.220000,78.775000,4.000000,0.000000,4.100000,12.100000,5.00000,1.000000
75%,2024.000000,9.000000,87.877500,2.000000,140.160000,10.000000,5.250000,6.990000,18.000000,13.990000,149.872500,5.000000,0.000000,4.400000,21.100000,9.00000,1.000000
max,2026.000000,12.000000,697.030000,5.000000,2636.450000,50.000000,421.580000,9.990000,20.000000,303.990000,2730.880000,14.000000,1.000000,5.000000,361.200000,24.00000,1.000000



product_summary


,total_orders,total_revenue_usd,avg_price,avg_rating,return_rate,avg_discount_pct,avg_delivery_days
count,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000
mean,168.171429,21135.674357,60.435286,3.999714,8.750000,5.582286,4.169214
std,113.141087,27013.447877,33.624582,0.095631,2.423283,0.825639,0.240506
min,38.000000,3231.830000,20.000000,3.600000,3.000000,3.310000,3.430000
25%,80.750000,7723.132500,31.132500,3.960000,7.000000,5.107500,4.047500
50%,142.500000,10362.085000,55.055000,4.010000,8.000000,5.580000,4.160000
75%,185.000000,16083.300000,80.447500,4.052500,10.000000,6.115000,4.272500
max,466.000000,124113.880000,143.340000,4.230000,17.000000,7.750000,4.860000



monthly_revenue


,year,month,orders,revenue_usd,avg_order_value,avg_discount_pct,return_rate,unique_customers,new_customers
count,75.000000,75.000000,75.000000,75.000000,75.000000,75.000000,75.0,75.000000,75.000000
mean,2022.640000,6.320000,273.293333,34468.353333,126.048533,5.632933,0.0,268.400000,97.120000
std,1.820603,3.522745,17.005383,3436.609880,8.841044,0.598473,0.0,16.604298,9.366357
min,2020.000000,1.000000,237.000000,25922.180000,109.380000,4.340000,0.0,236.000000,76.000000
25%,2021.000000,3.000000,262.000000,31878.580000,119.905000,5.220000,0.0,256.000000,93.000000
50%,2023.000000,6.000000,269.000000,34020.340000,125.540000,5.590000,0.0,263.000000,97.000000
75%,2024.000000,9.000000,285.500000,36749.420000,130.955000,6.185000,0.0,281.000000,102.000000
max,2026.000000,12.000000,316.000000,44793.180000,153.290000,7.060000,0.0,309.000000,122.000000


## Обработка пропусков

In [7]:
# Проверка природы пропусков customer_rating
print(orders.groupby('order_status')['customer_rating'].apply(lambda s: round(s.notna().mean(), 3)))

# Добавление флага has_rating
orders['has_rating'] = orders['customer_rating'].notna().astype(int)
print('has_rating:', orders['has_rating'].value_counts().to_dict())

order_status
Cancelled     0.000
Delivered     0.451
Processing    0.000
Returned      0.000
Name: customer_rating, dtype: float64
has_rating: {0: 15749, 1: 9251}


In [8]:
# customer_rating не импутируем: пропуск содержательный («оценки нет»)
orders['has_rating'] = orders['customer_rating'].notna().astype(int)
print('has_rating:', orders['has_rating'].value_counts().to_dict())

has_rating: {0: 15749, 1: 9251}


## Категориальные справочники

In [9]:
# Проверка уникальных значений категорий
cat_cols = {
    'customers': ['country', 'gender', 'membership_tier', 'preferred_category',
                  'preferred_device', 'preferred_payment_method', 'acquisition_channel'],
    'orders': ['category', 'payment_method', 'device_used', 'order_status'],
}
for table, cols in cat_cols.items():
    for col in cols:
        vals = frames[table][col].astype(str)
        norm = vals.str.strip().str.lower()
        flag = ' <-- СКРЫТЫЕ ДУБЛИ (регистр/пробелы)' if norm.duplicated().any() else ''
        print(f"{table}.{col}: {vals.nunique()} уникальных{flag}")

print('\norder_status:', sorted(frames['orders']['order_status'].unique()))
print('membership_tier:', sorted(frames['customers']['membership_tier'].unique()))

customers.country: 20 уникальных <-- СКРЫТЫЕ ДУБЛИ (регистр/пробелы)
customers.gender: 3 уникальных <-- СКРЫТЫЕ ДУБЛИ (регистр/пробелы)
customers.membership_tier: 4 уникальных <-- СКРЫТЫЕ ДУБЛИ (регистр/пробелы)
customers.preferred_category: 14 уникальных <-- СКРЫТЫЕ ДУБЛИ (регистр/пробелы)
customers.preferred_device: 3 уникальных <-- СКРЫТЫЕ ДУБЛИ (регистр/пробелы)
customers.preferred_payment_method: 7 уникальных <-- СКРЫТЫЕ ДУБЛИ (регистр/пробелы)
customers.acquisition_channel: 6 уникальных <-- СКРЫТЫЕ ДУБЛИ (регистр/пробелы)
orders.category: 14 уникальных <-- СКРЫТЫЕ ДУБЛИ (регистр/пробелы)
orders.payment_method: 7 уникальных <-- СКРЫТЫЕ ДУБЛИ (регистр/пробелы)
orders.device_used: 3 уникальных <-- СКРЫТЫЕ ДУБЛИ (регистр/пробелы)
orders.order_status: 4 уникальных <-- СКРЫТЫЕ ДУБЛИ (регистр/пробелы)

order_status: ['Cancelled', 'Delivered', 'Processing', 'Returned']
membership_tier: ['Free', 'Gold', 'Platinum', 'Silver']


## Даты и бизнес-правила

In [10]:
# Конвертация строк в datetime
customers['registration_date'] = pd.to_datetime(customers['registration_date'], errors='coerce')
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
orders['delivery_date'] = pd.to_datetime(orders['delivery_date'], errors='coerce')

# Проверка диапазонов
for label, s in [('registration_date', customers['registration_date']),
                 ('order_date', orders['order_date']),
                 ('delivery_date', orders['delivery_date'])]:
    print(f"{label}: min={s.min().date()}, max={s.max().date()}, NaT={s.isna().sum()}")

# Бизнес-правило: доставка не раньше заказа
print('delivery_date < order_date:', (orders['delivery_date'] < orders['order_date']).sum())

registration_date: min=2011-08-03, max=2026-04-01, NaT=0
order_date: min=2020-01-01, max=2026-03-30, NaT=0
delivery_date: min=2020-01-03, max=2026-04-12, NaT=0
delivery_date < order_date: 0


## Верификация гипотез о качестве данных

In [11]:
# Уникальность ключей
print('customer_id unique:', customers['customer_id'].is_unique)
print('order_id unique:', orders['order_id'].is_unique)
print('product key unique:', not product_summary.duplicated(subset=['category', 'product_name']).any())
print('month key unique:', not monthly_revenue.duplicated(subset=['year', 'month']).any())

# Покрытие периодов
o_periods = set(zip(orders['year'], orders['month']))
mr_periods = set(zip(monthly_revenue['year'], monthly_revenue['month']))
print('месяцы только в orders:', sorted(o_periods - mr_periods))
print('месяцы только в monthly_revenue:', sorted(mr_periods - o_periods))

# Объясняет ли покрытие расхождение 20 497 vs 25 000
mask = pd.Series(zip(orders['year'], orders['month'])).isin(mr_periods)
print('заказов в покрытии monthly_revenue:', mask.sum())

# Гипотеза о природе пропусков rating
print(orders.groupby('order_status')['customer_rating'].apply(lambda s: round(s.notna().mean(), 3)))

customer_id unique: True
order_id unique: True
product key unique: True
month key unique: True
месяцы только в orders: []
месяцы только в monthly_revenue: []
заказов в покрытии monthly_revenue: 25000
order_status
Cancelled     0.000
Delivered     0.451
Processing    0.000
Returned      0.000
Name: customer_rating, dtype: float64


In [12]:
# Гипотеза 1: product_summary = все заказы кроме Cancelled
nc = orders[orders['order_status'] != 'Cancelled']
ps = product_summary.set_index(['category', 'product_name'])
agg_o = nc.groupby(['category', 'product_name']).size()
agg_r = nc.groupby(['category', 'product_name'])['total_amount_usd'].sum().round(2)
print('orders match:', (ps['total_orders'] == agg_o).all())
print('revenue match:', (abs(ps['total_revenue_usd'] - agg_r) <= 0.01).all())

# Гипотеза 2: 716 клиентов вне окна monthly_revenue
reg = customers['registration_date'].dt.to_period('M')
in_window = (reg >= pd.Period('2020-01')) & (reg <= pd.Period('2026-03'))
print('регистраций в окне:', in_window.sum(), '| вне окна:', (~in_window).sum())
print('sum new_customers:', monthly_revenue['new_customers'].sum())

orders match: True
revenue match: True
регистраций в окне: 7809 | вне окна: 191
sum new_customers: 7284


## Производные признаки

In [13]:
# Производные признаки
ref_date = orders['order_date'].max()
customers['registration_month'] = customers['registration_date'].dt.to_period('M').astype(str)
customers['tenure_days'] = (ref_date - customers['registration_date']).dt.days

# Арифметические проверки
sub = (orders['unit_price_usd'] * orders['quantity']).round(2)
disc = (orders['subtotal_usd'] * orders['discount_pct'] / 100).round(2)
total = (orders['subtotal_usd'] - orders['discount_amount_usd']
         + orders['shipping_fee_usd'] + orders['tax_amount_usd']).round(2)

print('subtotal mismatch:', (abs(sub - orders['subtotal_usd']) > 0.01).sum())
print('discount mismatch:', (abs(disc - orders['discount_amount_usd']) > 0.01).sum())
print('total mismatch:', (abs(total - orders['total_amount_usd']) > 0.01).sum())

subtotal mismatch: 0
discount mismatch: 0
total mismatch: 0


## Проверка гипотезы: monthly_revenue агрегирует только Delivered

In [14]:
st = orders['order_status'].value_counts()
print(st)
print('сумма orders в monthly_revenue:', monthly_revenue['orders'].sum())
print('выручка Delivered:', round(orders.loc[orders['order_status'] == 'Delivered', 'total_amount_usd'].sum(), 2))
print('выручка monthly_revenue:', round(monthly_revenue['revenue_usd'].sum(), 2))

order_status
Delivered     20497
Returned       2020
Cancelled      1456
Processing     1027
Name: count, dtype: int64
сумма orders в monthly_revenue: 20497
выручка Delivered: 2585126.5
выручка monthly_revenue: 2585126.5


## Наблюдения по качеству данных
Краткая сводка; полный отчёт: `reports/data_quality_report.md`
- Пропуски: только orders.customer_rating (15 749, 63,0% строк) — не импутируем, флаг has_rating
- Дубликаты: 0 во всех таблицах
- monthly_revenue.return_rate = 0 во всех строках — поле исключено из анализа
- monthly_revenue агрегирует только заказы со статусом Delivered (подтверждено)

## Сохранение результатов

In [15]:
# Сохранение очищенных данных
OUT = '../data/processed/'
customers.to_csv(f'{OUT}customers_clean.csv', index=False)
orders.to_csv(f'{OUT}orders_clean.csv', index=False)
product_summary.to_csv(f'{OUT}products_clean.csv', index=False)
monthly_revenue.to_csv(f'{OUT}revenue_monthly_clean.csv', index=False)
quality.to_csv(f'{OUT}data_quality_summary.csv', index=True)
print('сохранено в data/processed/')

сохранено в data/processed/


In [16]:
import sys
print(sys.executable)

/Users/macpro/Documents/projects/ecom_customer_behavior_2020_2026/.venv/bin/python
